TODO - посмотреть какие шрифты можно добавить
<link rel="stylesheet" type="text/css" href="path/to/custom.css">

hello

In [ ]:
# Это краткий конспект полезных выводов из курса по нейросетям - https://habr.com/ru/articles/414165/
# Также - полезная статья Яндекса со ссылками на источники - https://habr.com/ru/companies/yandex/articles/307260/

### Общая теория

Задача ML = задача оптимизации весов модели на тренировочной выборке для лучшего предикта отложенной.  
- Пусть у нас есть объекты u1, u2, ... (например, клиенты)
- Каждый объект может быть описан признаками в N-мерном пр-ве u1 = x11, x12, ... x1N; ...
- Мы решаем задачу классификации -> определяем какому из классов принадлежат объекты: y1, ... yM

Есть тренировочный размеченный датасет Train = u1, u2...  
Мы обучаем модель типа y = Q(w1, w2, ... wK, x1, ... xN) с какими то весами  
А далее делаем предсказание на отложенной выборке predict(Q, u) -> y_predict

**Линейный классификатор**: y = (w, x)  
Или в матричном виде: W_NM * X_N = Y_M  
Здесь - X_N столбец признаков объекта, Y_M - столбец вероятностей принадлежности к каждому классу  


___

**Метод макс правдоподобия**  
Макс. правдоподобие - общая метрика, которую надо оптимизировать для поиска корректных весов модели  

**Likelihood** = p(y=y_train_1 | u_train_1) * ... * p(y=y_train_D | u_train_D)  
Пояснение: это вероятность того что мы получим верный y из возможных M классов для каждого объекта из Train  

Вероятность получить y_i для каждого u = x1, x2... xN задается через модель и ее веса **w**  

Для оптимальных весов необходимо Likelihood -> MAX  
Это эквивалентно -ln(Likelihood) -> MIN  
Или **LossFunction** = -Sum(ln(p(y = y_train | u_train))) -> MIN  

Для поиска минимума функции потерь необходимо считать ее градиент по весам w и идти по -grad(LossFunc)

___


**Регуляризация**  
Для линейного классификатора можно написать LossFunc = L(w, x) -> grad(LossFunc, w)  
Однако минимум Lossfunc может быть найден при разных значениях w (нет единственного решения).  
Поэтому чтобы это исправить и норм оптимизировать - делают **регуляризацию**

LossFunc = -sum(ln(p)) + lambda * R(w)  
Варианты функции R:
1) L1: R = |w1| + |w2| + ...
2) L2: R^2 = w1^2 + w2^2 + ...  
L2 метрика хорошо дифференциирует и помимо добавления однозначности - наказывает модель за очень большие w_i  
Оказывается, плавное распределение значений весов w_1, w_2 ... положительно сказывается на том, чтобы  
модель не переобучалась (то есть не искала слишком частных решений которые фитят Train).  

Итак, с помощью регуляризации получаем единственную точку для оптимизации, страхуемся от переобучения  
lambda - фактор, задающий насколько сильно наказываем модель за избыточные веса. Подбирается как гиперпараметр.

### Нейронные сети

В линейном классификаторе мы по сути делаем следующие преобразования:  
x_ = (x1, ... xN)  ; x_ * w_NM = y_M  

Для нейросетей используем следующие цепочки преобразований:   
tmp = x_ * w_N_K1 # на выходе получим столбец величины K1  
tmp = L(tmp) # применяем некоторое нелинейное преобразование (функция активации)  
tmp = tmp * w_K1_K2 # очередное "взвешивание" матрицей - на выходе столбец K2  
tmp = L(tmp)  
...  
tmp = tmp * w_Kx_M = y_M # здесь получаем столбец вероятностей принадлежности к классам  

каждое умножение на матрицу w_x_y -> это новый **слой** обучаемой нейросети  
Функция активации L(x) нелинейная, чтобы получаемая модель Q была более гибкой и настраиваемой.  
Один из эффективных вариантов для обучения: L = Relu(x) = x if x > 0 else 0.  


Почему сеть нейронная. Визуально .... TODO дописать

____

Итак, мы получаем модель с рядом слоев, каждый из которых содержит K_x * K_y весов в общем случае.  
Так как в каждом из слоев - линейное умножение, то можно аналитически записать LikeliHood модели.  
Далее, также как и в другой модели - считаем grad(LossFunc), смещаем веса в направлении -grad и двигаемся  
иттеративно пока не найдем минимум функции. Данные веса w - веса обученной нейросети

При этом расчет градиентов на каждом шаге сводится к матричным операциям со слоями и хорошо параллелится  
в плане вычислений. Поэтому обучение сетей производится на видеокартах

### Сверточные нейронные сети

Проблема классических сетей в огромном кол-ве обучаемых параметров  
Каждый слой - это по сути матрица которая взвешивает каждый элемент 